# 04. Sensor Ablation Study & Feature Importance
## Evaluating Multi-Sensor Contribution & Permutation Importance

This notebook evaluates the individual and joint contributions of each sensor group (AS7341, TCS34725, VL53L1X) and quantifies permutation feature importance.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, "..")
from src.features.preprocessing import load_dataset, get_X_y_groups
from src.data.splitting import subject_level_train_test_split
from src.models.ablation import run_ablation_study
from sklearn.inspection import permutation_importance
import joblib

%matplotlib inline
sns.set_theme(style="whitegrid")


### 1. Execute Sensor Ablation Study


In [ ]:
df = load_dataset()
df_train, df_test = subject_level_train_test_split(df)
df_ablation = run_ablation_study(df_train, df_test)
df_ablation


### 2. Visualize Ablation Results


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=df_ablation, x="config", y="f1_macro", palette="viridis")
plt.title("Sensor Ablation Study: Macro F1 by Sensor Configuration", fontsize=13)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Macro F1 Score")
plt.ylim(0, 1.0)
plt.tight_layout()
plt.show()


### 3. Permutation Feature Importance


In [ ]:
model_path = os.path.join("..", "models", "svm_final_pipeline.joblib")
pipeline = joblib.load(model_path)
X_test, y_test, _ = get_X_y_groups(df_test)

perm_res = permutation_importance(pipeline, X_test, y_test, scoring="f1_macro", n_repeats=10, random_state=42)
perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm_res.importances_mean,
    "importance_std": perm_res.importances_std
}).sort_values("importance_mean", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=perm_df, x="importance_mean", y="feature", palette="magma")
plt.title("Permutation Feature Importance (Test Set Macro F1 Drop)", fontsize=13)
plt.xlabel("Mean Decrease in Macro F1")
plt.tight_layout()
plt.show()
